In [ ]:
import os
import json
import warnings
import numpy as np
import xarray as xr
import proplot as pplt
warnings.filterwarnings('ignore')
pplt.rc.update({'figure.dpi':100})

In [ ]:
with open('../scripts/configs.json','r',encoding='utf-8') as f:
    CONFIGS = json.load(f)

SPLITSDIR   = CONFIGS['filepaths']['splits']
PREDSDIR    = CONFIGS['filepaths']['predictions']
WEIGHTSDIR  = CONFIGS['filepaths']['weights']
INTERIMDIR  = CONFIGS['filepaths']['interim']
LATRANGE    = CONFIGS['domain']['latrange']
LONRANGE    = CONFIGS['domain']['lonrange']
SEEDS       = CONFIGS['experiments']['nn']['seeds']
SPLIT       = 'valid'

FIELDVARS   = ['rh','thetae','thetaestar']
FIELDLABELS = {'rh':'RH','thetae':'$\\theta_e$','thetaestar':'$\\theta_e^*$'}

RUNS = {
    'nonparametric': 'nonparametric_rh_thetae_thetaestar_lf_lhf_shf',
    'lfconditioned': 'lfconditioned_rh_thetae_thetaestar_lf_lhf_shf',
    'lfthreshold':   'lfthreshold_rh_thetae_thetaestar_lf_lhf_shf'}

LABELS = {
    'nonparametric': 'Universal',
    'lfconditioned': 'LF-Conditioned',
    'lfthreshold':   'LF-Threshold'}

In [ ]:
# Load kernel weights for each model
# nonparametric: k1 = universal kernel (nfieldvars, nlevs)
# lfconditioned: k1 = ocean endpoint (lf=0), k2 = land endpoint (lf=1)
# lfthreshold:   k1 = ocean kernel, k2 = land kernel

weights = {}
for key,runname in RUNS.items():
    seed = SEEDS[0]
    wpath = os.path.join(WEIGHTSDIR,f'{runname}_{seed}_weights.nc')
    if not os.path.exists(wpath):
        print(f'Missing: {wpath}')
        continue
    ds = xr.open_dataset(wpath,engine='h5netcdf')
    w = {'k1':ds['k1'].isel(seed=0).values}  # (nfieldvars, nlevs)
    if 'k2' in ds:
        w['k2'] = ds['k2'].isel(seed=0).values
    w['sig'] = ds['sig'].values if 'sig' in ds.coords else np.arange(w['k1'].shape[1])
    ds.close()
    weights[key] = w
    nk = len([k for k in w if k.startswith('k')])
    print(f'{key}: {nk} component(s), shape {w["k1"].shape}')

In [ ]:
# Learned kernel weight profiles by field variable
# One column per field variable, one row per model
# For models with land/ocean kernels, both are shown

nmodels = len(weights)
nfields = len(FIELDVARS)
fig,axs = pplt.subplots(nrows=nmodels,ncols=nfields,refwidth=2,refheight=2.2,
                        sharex=True,sharey=True)
axs.format(suptitle='Learned Kernel Weights',ylabel='$\\sigma$',xlabel='Weight',
           yreverse=True,ylim=(1.0,0.5))

for i,(key,w) in enumerate(weights.items()):
    sig = w['sig']
    for j,var in enumerate(FIELDVARS):
        ax = axs[i,j] if nmodels>1 else axs[j]
        if 'k2' in w:
            ax.plot(w['k1'][j],sig,color='blue5',linewidth=1.5,label='Ocean')
            ax.plot(w['k2'][j],sig,color='red5',linewidth=1.5,label='Land')
        else:
            ax.plot(w['k1'][j],sig,color='gray7',linewidth=1.5,label='Universal')
        if i==0:
            ax.format(title=FIELDLABELS[var])
        if j==0:
            ax.text(-0.35,0.5,LABELS[key],transform=ax.transAxes,
                    rotation=90,va='center',ha='center',fontsize=10)
        if i==0 and j==nfields-1:
            ax.legend(loc='ur',ncols=1)

pplt.show()

In [ ]:
# Overlay all models on one panel per field variable
# Shows how land/ocean kernels diverge from the universal kernel

fig,axs = pplt.subplots(nrows=1,ncols=nfields,refwidth=2.5,refheight=3,sharey=True)
axs.format(suptitle='Kernel Comparison: Universal vs. Land/Ocean',
           ylabel='$\\sigma$',xlabel='Weight',yreverse=True,ylim=(1.0,0.5))

for j,var in enumerate(FIELDVARS):
    ax = axs[j]
    # Universal
    if 'nonparametric' in weights:
        w = weights['nonparametric']
        ax.plot(w['k1'][j],w['sig'],color='gray5',linewidth=2,label='Universal')
    # LF-Conditioned endpoints
    if 'lfconditioned' in weights:
        w = weights['lfconditioned']
        ax.plot(w['k1'][j],w['sig'],color='blue5',linewidth=1.5,linestyle='--',label='Cond. Ocean')
        ax.plot(w['k2'][j],w['sig'],color='red5',linewidth=1.5,linestyle='--',label='Cond. Land')
    # LF-Threshold kernels
    if 'lfthreshold' in weights:
        w = weights['lfthreshold']
        ax.plot(w['k1'][j],w['sig'],color='blue5',linewidth=1.5,linestyle=':',label='Thresh. Ocean')
        ax.plot(w['k2'][j],w['sig'],color='red5',linewidth=1.5,linestyle=':',label='Thresh. Land')
    ax.format(title=FIELDLABELS[var])
    if j==nfields-1:
        ax.legend(loc='ur',ncols=1)

pplt.show()

In [ ]:
# Land minus ocean kernel difference
# Positive = land kernel puts more weight at that level

diff_models = {k:v for k,v in weights.items() if 'k2' in v}
if diff_models:
    fig,axs = pplt.subplots(nrows=1,ncols=nfields,refwidth=2.5,refheight=3,sharey=True)
    axs.format(suptitle='Land $-$ Ocean Kernel Difference',
               ylabel='$\\sigma$',xlabel='$\\Delta$ Weight',yreverse=True,ylim=(1.0,0.5))

    styles = {'lfconditioned':('--','LF-Conditioned'),'lfthreshold':(':','LF-Threshold')}
    for j,var in enumerate(FIELDVARS):
        ax = axs[j]
        ax.axvline(0,color='gray5',linewidth=0.8,linestyle='-')
        for key,w in diff_models.items():
            diff = w['k2'][j]-w['k1'][j]
            ls,label = styles[key]
            ax.plot(diff,w['sig'],color='purple5',linewidth=1.5,linestyle=ls,label=label)
        ax.format(title=FIELDLABELS[var])
        if j==nfields-1:
            ax.legend(loc='ur',ncols=1)

    pplt.show()

In [ ]:
# Load predictions and observations

with xr.open_dataset(os.path.join(SPLITSDIR,f'{SPLIT}.h5'),engine='h5netcdf') as ds:
    truetp = ds.tp.load()

lf = xr.open_dataarray(os.path.join(INTERIMDIR,'lf.nc'),engine='h5netcdf').load()

preds = {}
for key,runname in RUNS.items():
    filepath = os.path.join(PREDSDIR,f'{runname}_{SPLIT}_predictions.nc')
    if not os.path.exists(filepath):
        print(f'Missing: {filepath}')
        continue
    with xr.open_dataset(filepath) as ds:
        preds[key] = ds.tp.load()
    print(f'{key}: {preds[key].shape}')

In [ ]:
# Time-mean bias maps for each model

fig,axs = pplt.subplots(nrows=1,ncols=len(preds),proj='cyl',refwidth=2.5,refheight=2.5)
axs.format(suptitle=f'Time-Mean Precipitation Bias ({SPLIT})',
           coast=True,grid=False,latlim=LATRANGE,lonlim=LONRANGE)

biases = {}
for key,predtp in preds.items():
    ytrue,ypred = xr.align(truetp,predtp,join='inner')
    ypredmean = ypred.mean('seed') if 'seed' in ypred.dims else ypred
    biases[key] = (ypredmean-ytrue).mean('time').squeeze()

vmax = max(float(abs(b).max()) for b in biases.values())
for i,(key,bias) in enumerate(biases.items()):
    im = axs[i].pcolormesh(bias.lon,bias.lat,bias,cmap='ColdHot_r',
                           vmin=-vmax,vmax=vmax,levels=21,extend='both')
    axs[i].format(title=LABELS[key])

fig.colorbar(im,loc='r',label='Bias (mm)')
pplt.show()

In [ ]:
# Bias split by land vs ocean
# Bar chart: mean bias over land (lf >= 0.5) and ocean (lf < 0.5) for each model

is_land  = lf>=0.5
is_ocean = lf<0.5

fig,ax = pplt.subplots(refwidth=5,refheight=3)
ax.format(suptitle=f'Mean Bias: Land vs. Ocean ({SPLIT})',ylabel='Mean Bias (mm)',grid=False)

x = np.arange(len(preds))
width = 0.35
land_biases,ocean_biases,labels = [],[],[]
for key,predtp in preds.items():
    ytrue,ypred = xr.align(truetp,predtp,join='inner')
    ypredmean = ypred.mean('seed') if 'seed' in ypred.dims else ypred
    diff = ypredmean-ytrue
    land_biases.append(float(diff.where(is_land).mean(skipna=True)))
    ocean_biases.append(float(diff.where(is_ocean).mean(skipna=True)))
    labels.append(LABELS[key])

ax.bar(x-width/2,land_biases,width=width,color='red5',label='Land')
ax.bar(x+width/2,ocean_biases,width=width,color='blue5',label='Ocean')
ax.axhline(0,color='k',linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.legend(loc='ur')
pplt.show()

In [ ]:
# R2 split by land vs ocean

def get_r2(ytrue,ypred,dims=None):
    dims  = list(ytrue.dims) if dims is None else dims
    ssres = ((ytrue-ypred)**2).sum(dim=dims,skipna=True)
    sstot = ((ytrue-ytrue.mean(dim=dims,skipna=True))**2).sum(dim=dims,skipna=True)
    return 1-ssres/sstot

fig,ax = pplt.subplots(refwidth=5,refheight=3)
ax.format(suptitle=f'R$^2$: Land vs. Ocean ({SPLIT})',ylabel='R$^2$',grid=False)

x = np.arange(len(preds))
width = 0.35
land_r2s,ocean_r2s,labels = [],[],[]
for key,predtp in preds.items():
    ytrue,ypred = xr.align(truetp,predtp,join='inner')
    ypredmean = ypred.mean('seed') if 'seed' in ypred.dims else ypred
    land_r2  = get_r2(ytrue.where(is_land),ypredmean.where(is_land))
    ocean_r2 = get_r2(ytrue.where(is_ocean),ypredmean.where(is_ocean))
    land_r2s.append(float(land_r2))
    ocean_r2s.append(float(ocean_r2))
    labels.append(LABELS[key])

ax.bar(x-width/2,land_r2s,width=width,color='red5',label='Land')
ax.bar(x+width/2,ocean_r2s,width=width,color='blue5',label='Ocean')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.legend(loc='ur')
pplt.show()

In [ ]:
# Spatial R2 improvement over universal kernel
# Positive = conditioned/threshold model is better

if 'nonparametric' in preds:
    ytrue_ref,ypred_ref = xr.align(truetp,preds['nonparametric'],join='inner')
    ypred_ref = ypred_ref.mean('seed') if 'seed' in ypred_ref.dims else ypred_ref
    r2_ref = get_r2(ytrue_ref,ypred_ref,dims=['time']).squeeze()

    compare = {k:v for k,v in preds.items() if k!='nonparametric'}
    fig,axs = pplt.subplots(nrows=1,ncols=len(compare),proj='cyl',refwidth=3,refheight=2.5)
    axs.format(suptitle=f'$\\Delta$R$^2$ vs. Universal Kernel ({SPLIT})',
               coast=True,grid=False,latlim=LATRANGE,lonlim=LONRANGE)

    for i,(key,predtp) in enumerate(compare.items()):
        ytrue_c,ypred_c = xr.align(truetp,predtp,join='inner')
        ypred_c = ypred_c.mean('seed') if 'seed' in ypred_c.dims else ypred_c
        r2_c = get_r2(ytrue_c,ypred_c,dims=['time']).squeeze()
        dr2  = r2_c-r2_ref
        im = axs[i].pcolormesh(dr2.lon,dr2.lat,dr2,cmap='ColdHot',
                               vmin=-0.1,vmax=0.1,levels=21,extend='both')
        axs[i].format(title=LABELS[key])

    fig.colorbar(im,loc='r',label='$\\Delta$R$^2$')
    pplt.show()